## Imports:

In [1]:
import numpy as np
import pandas as pd
from scripts.utilities import *
from scripts.features import *
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from scipy.stats import pointbiserialr

## Reading in data:

In [2]:
consumers, accounts, transactions = get_data()
transactions.amount = transactions.amount.apply(abs)

Data successfully loaded and processed.


## Preprocesssing:

In [3]:
consumers.evaluation_date = pd.to_datetime(consumers.evaluation_date)
accounts.balance_date = pd.to_datetime(accounts.balance_date)
transactions.posted_date = pd.to_datetime(transactions.posted_date)

In [4]:
consumers.prism_consumer_id = consumers.prism_consumer_id.astype(int)
accounts.prism_consumer_id = accounts.prism_consumer_id.astype(int)
transactions.prism_consumer_id = transactions.prism_consumer_id.astype(int)

In [5]:
accounts.prism_account_id = accounts.prism_account_id.astype(int)

In [6]:
transactions.amount = transactions.amount.apply(abs)

## EDA:

In [7]:
df = pd.DataFrame(transactions.category.value_counts(normalize=True) * 100).reset_index()
df.iloc[:10]

,category,proportion
0,FOOD_AND_BEVERAGES,13.903986
1,GENERAL_MERCHANDISE,12.320750
2,SELF_TRANSFER,10.878478
3,EXTERNAL_TRANSFER,10.737046
4,MISCELLANEOUS,9.738813
5,AUTOMOTIVE,7.406044
6,GROCERIES,4.912989
7,ENTERTAINMENT,3.917644
8,CREDIT_CARD_PAYMENT,2.899293
9,PAYCHECK,2.812533


In [8]:
c_df = consumers.dropna(subset='DQ_TARGET')
a_df = accounts[accounts.prism_consumer_id.isin(c_df.prism_consumer_id)]
t_df = transactions[transactions.prism_consumer_id.isin(c_df.prism_consumer_id)]

display(c_df, a_df, t_df)

,prism_consumer_id,evaluation_date,credit_score,DQ_TARGET
0,0,2021-09-01,726.0,0.0
1,1,2021-07-01,626.0,0.0
2,2,2021-05-01,680.0,0.0
3,3,2021-03-01,734.0,0.0
4,4,2021-10-01,676.0,0.0
...,...,...,...,...
13995,13995,2022-01-22,802.0,0.0
13996,13996,2022-02-01,652.0,0.0
13997,13997,2021-12-24,765.0,0.0
13998,13998,2022-01-30,685.0,0.0


,prism_consumer_id,prism_account_id,account_type,balance_date,balance
0,3023,0,SAVINGS,2021-08-31,90.57
1,3023,1,CHECKING,2021-08-31,225.95
5,3920,5,SAVINGS,2021-10-31,0.26
6,3920,6,CHECKING,2021-10-31,4.42
7,3920,7,CHECKING,2021-10-31,1.05
...,...,...,...,...,...
24461,11500,24461,CHECKING,2022-03-27,732.75
24462,11615,24462,SAVINGS,2022-03-30,5.00
24463,11615,24463,CHECKING,2022-03-30,1956.46
24464,12210,24464,CHECKING,2022-03-28,2701.51


,prism_consumer_id,prism_transaction_id,amount,credit_or_debit,posted_date,category
0,3023,0,0.05,CREDIT,2021-04-16,MISCELLANEOUS
1,3023,1,481.56,CREDIT,2021-04-30,LOAN
2,3023,2,0.05,CREDIT,2021-05-16,MISCELLANEOUS
3,3023,3,0.07,CREDIT,2021-06-16,MISCELLANEOUS
4,3023,4,0.06,CREDIT,2021-07-16,MISCELLANEOUS
...,...,...,...,...,...,...
6407316,10533,6405304,4.96,DEBIT,2022-03-11,BILLS_UTILITIES
6407317,10533,6405305,63.48,DEBIT,2022-03-30,LOAN
6407318,10533,6405306,53.99,DEBIT,2022-03-30,LOAN
6407319,10533,6405307,175.98,DEBIT,2022-03-31,LOAN


In [9]:
t_df.category.value_counts()

category
FOOD_AND_BEVERAGES       713779
GENERAL_MERCHANDISE      633497
SELF_TRANSFER            553172
EXTERNAL_TRANSFER        548235
MISCELLANEOUS            503638
AUTOMOTIVE               382582
GROCERIES                253635
ENTERTAINMENT            202670
CREDIT_CARD_PAYMENT      149348
PAYCHECK                 143859
LOAN                      99404
DEPOSIT                   91325
TRANSPORATION             86233
ATM_CASH                  84661
BNPL                      76375
HEALTHCARE_MEDICAL        64671
ESSENTIAL_SERVICES        59637
INVESTMENT                54574
HOME_IMPROVEMENT          49494
REFUND                    47522
ACCOUNT_FEES              45432
INSURANCE                 43665
BILLS_UTILITIES           37003
TRAVEL                    24779
INVESTMENT_INCOME         17353
GAMBLING                  17172
BANKING_CATCH_ALL         16566
FITNESS                   14567
PETS                      13674
OVERDRAFT                 13149
TAX                       13105

In [10]:
y_train = c_df.set_index('prism_consumer_id')['DQ_TARGET']

In [11]:
date_train = c_df.set_index('prism_consumer_id')['evaluation_date'].to_dict()

In [12]:
c_df.DQ_TARGET.value_counts(normalize=True) * 100

DQ_TARGET
0.0    91.616667
1.0     8.383333
Name: proportion, dtype: float64

- Observed that not all consumers have any transactional data:

In [13]:
test = c_df[~c_df.prism_consumer_id.isin(t_df.prism_consumer_id.unique())]

display(test)
test.DQ_TARGET.value_counts()

,prism_consumer_id,evaluation_date,credit_score,DQ_TARGET
5003,5003,2023-07-26,601.0,0.0
5007,5007,2023-08-09,523.0,0.0
5024,5024,2023-03-18,427.0,0.0
5036,5036,2023-04-29,632.0,0.0
5044,5044,2023-03-01,661.0,0.0
...,...,...,...,...
12780,12780,2022-02-07,653.0,0.0
12932,12932,2022-03-19,704.0,0.0
13479,13479,2022-01-09,645.0,0.0
13607,13607,2022-01-10,627.0,0.0


DQ_TARGET
0.0    341
1.0     58
Name: count, dtype: int64

## Inflows vs Outflows:
- credit = inflow 
- debit = outflow

In [14]:
inflows = t_df[t_df.credit_or_debit == 'CREDIT']
outflows = t_df[t_df.credit_or_debit == 'DEBIT']

# display(inflows, outflows)

# Using transactional data:
- Would help to look at income and spending habits:
- Only 47/50 categories are relevant

In [15]:
t_df.category.unique()

array(['MISCELLANEOUS', 'LOAN', 'EXTERNAL_TRANSFER', 'DEPOSIT',
       'SELF_TRANSFER', 'INVESTMENT', 'PAYCHECK', 'REFUND',
       'ENTERTAINMENT', 'FOOD_AND_BEVERAGES', 'GROCERIES', 'FITNESS',
       'GENERAL_MERCHANDISE', 'HEALTHCARE_MEDICAL', 'GAMBLING',
       'GIFTS_DONATIONS', 'INSURANCE', 'CREDIT_CARD_PAYMENT', 'RENT',
       'AUTOMOTIVE', 'HOME_IMPROVEMENT', 'TAX', 'BILLS_UTILITIES',
       'ESSENTIAL_SERVICES', 'ATM_CASH', 'TRANSPORATION', 'EDUCATION',
       'PENSION', 'TRAVEL', 'PETS', 'MORTGAGE', 'BANKING_CATCH_ALL',
       'DEBT', 'BNPL', 'AUTO_LOAN', 'GOVERNMENT_SERVICES',
       'CORPORATE_PAYMENTS', 'LEGAL', 'RISK_CATCH_ALL', 'OTHER_BENEFITS',
       'UNEMPLOYMENT_BENEFITS', 'ACCOUNT_FEES', 'TIME_OR_STUFF',
       'RTO_LTO', 'CHILD_DEPENDENTS', 'OVERDRAFT', 'INVESTMENT_INCOME'],
      dtype=object)

In [16]:
# t_df.groupby(['category', 'credit_or_debit'])[['amount']].count()

### Method 1: Determining income by using the obvious categories

In [17]:
income = ['PAYCHECK', 'PENSION', 'OTHER_BENEFITS', 'UNEMPLOYMENT_BENEFITS', 'INVESTMENT_INCOME']

In [18]:
t_df['is_income'] = np.where(
    (t_df['credit_or_debit'] == 'CREDIT') & (t_df['category'].isin(income)),
    True,
    False
)
t_df

C:\Users\bdion\AppData\Local\Temp\ipykernel_1636\3499483718.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  t_df['is_income'] = np.where(


,prism_consumer_id,prism_transaction_id,amount,credit_or_debit,posted_date,category,is_income
0,3023,0,0.05,CREDIT,2021-04-16,MISCELLANEOUS,False
1,3023,1,481.56,CREDIT,2021-04-30,LOAN,False
2,3023,2,0.05,CREDIT,2021-05-16,MISCELLANEOUS,False
3,3023,3,0.07,CREDIT,2021-06-16,MISCELLANEOUS,False
4,3023,4,0.06,CREDIT,2021-07-16,MISCELLANEOUS,False
...,...,...,...,...,...,...,...
6407316,10533,6405304,4.96,DEBIT,2022-03-11,BILLS_UTILITIES,False
6407317,10533,6405305,63.48,DEBIT,2022-03-30,LOAN,False
6407318,10533,6405306,53.99,DEBIT,2022-03-30,LOAN,False
6407319,10533,6405307,175.98,DEBIT,2022-03-31,LOAN,False


### Method 2: Determining income by finding recurrence (and obvious categories)
-- Sorting by time first rather than grouping transactions by category first

In [19]:
recurrence_df = inflows.sort_values(['prism_consumer_id', 'posted_date'], ignore_index=True)
recurrence_df

,prism_consumer_id,prism_transaction_id,amount,credit_or_debit,posted_date,category
0,0,136703,1400.00,CREDIT,2021-03-17,TAX
1,0,136696,0.09,CREDIT,2021-03-19,MISCELLANEOUS
2,0,136704,1000.22,CREDIT,2021-03-19,PAYCHECK
3,0,136705,0.38,CREDIT,2021-03-19,MISCELLANEOUS
4,0,136706,1075.31,CREDIT,2021-04-06,PAYCHECK
...,...,...,...,...,...,...
878824,13999,6274780,4.00,CREDIT,2022-01-18,SELF_TRANSFER
878825,13999,6274781,16.00,CREDIT,2022-01-19,SELF_TRANSFER
878826,13999,6274846,200.00,CREDIT,2022-01-20,DEPOSIT
878827,13999,6274782,1.00,CREDIT,2022-01-21,SELF_TRANSFER


- Difference between dates:

In [20]:
time_diffs = recurrence_df.groupby('prism_consumer_id')['posted_date'].apply(
    lambda x: np.ediff1d(x.astype('int64') // 10**9, to_begin=0)
)
recurrence_df['time_diff'] = time_diffs.explode().astype(int).reset_index()['posted_date']
recurrence_df['time_diff'] = pd.to_timedelta(recurrence_df['time_diff'], unit='s')

- Difference between amounts:

In [21]:
amount_diffs = recurrence_df.groupby('prism_consumer_id')['amount'].apply(
    lambda x: abs(np.ediff1d(x, to_begin=0))
)
recurrence_df['amount_diff'] = amount_diffs.explode().reset_index()['amount']

In [22]:
recurrence_df

,prism_consumer_id,prism_transaction_id,amount,credit_or_debit,posted_date,category,time_diff,amount_diff
0,0,136703,1400.00,CREDIT,2021-03-17,TAX,0 days,0.0
1,0,136696,0.09,CREDIT,2021-03-19,MISCELLANEOUS,2 days,1399.91
2,0,136704,1000.22,CREDIT,2021-03-19,PAYCHECK,0 days,1000.13
3,0,136705,0.38,CREDIT,2021-03-19,MISCELLANEOUS,0 days,999.84
4,0,136706,1075.31,CREDIT,2021-04-06,PAYCHECK,18 days,1074.93
...,...,...,...,...,...,...,...,...
878824,13999,6274780,4.00,CREDIT,2022-01-18,SELF_TRANSFER,4 days,3.0
878825,13999,6274781,16.00,CREDIT,2022-01-19,SELF_TRANSFER,1 days,12.0
878826,13999,6274846,200.00,CREDIT,2022-01-20,DEPOSIT,1 days,184.0
878827,13999,6274782,1.00,CREDIT,2022-01-21,SELF_TRANSFER,1 days,199.0


- Setting thresholds for the time and amount differences to mark transactions as income:

In [23]:
recurrence_df['percent_change'] = recurrence_df['amount_diff'] / recurrence_df['amount'].shift(1).fillna(1) * 100

# Check if differences are within 10-20%:
recurrence_df['amount_threshold'] = recurrence_df['percent_change'].between(10, 20)

In [24]:
# Check if differences are within 30 days:
recurrence_df['days_threshold'] = recurrence_df.time_diff <= pd.Timedelta(days=30)
recurrence_df

,prism_consumer_id,prism_transaction_id,amount,credit_or_debit,posted_date,category,time_diff,amount_diff,percent_change,amount_threshold,days_threshold
0,0,136703,1400.00,CREDIT,2021-03-17,TAX,0 days,0.0,0.0,False,True
1,0,136696,0.09,CREDIT,2021-03-19,MISCELLANEOUS,2 days,1399.91,99.993571,False,True
2,0,136704,1000.22,CREDIT,2021-03-19,PAYCHECK,0 days,1000.13,1111255.555556,False,True
3,0,136705,0.38,CREDIT,2021-03-19,MISCELLANEOUS,0 days,999.84,99.962008,False,True
4,0,136706,1075.31,CREDIT,2021-04-06,PAYCHECK,18 days,1074.93,282876.315789,False,True
...,...,...,...,...,...,...,...,...,...,...,...
878824,13999,6274780,4.00,CREDIT,2022-01-18,SELF_TRANSFER,4 days,3.0,300.0,False,True
878825,13999,6274781,16.00,CREDIT,2022-01-19,SELF_TRANSFER,1 days,12.0,300.0,False,True
878826,13999,6274846,200.00,CREDIT,2022-01-20,DEPOSIT,1 days,184.0,1150.0,False,True
878827,13999,6274782,1.00,CREDIT,2022-01-21,SELF_TRANSFER,1 days,199.0,99.5,False,True


In [25]:
recurrence_df['is_income'] = np.where(
    (
        (recurrence_df.amount_threshold == True) &
        (recurrence_df.days_threshold == True)
    ) |
    (recurrence_df.category.isin(income)),
    True,
    False
)

In [26]:
income_df = recurrence_df.groupby(['prism_consumer_id', 'is_income'])[['amount']].sum().loc[pd.IndexSlice[:, True], :].reset_index()
income_df

,prism_consumer_id,is_income,amount
0,0,True,8860.56
1,1,True,11918.64
2,2,True,60.34
3,3,True,10147.81
4,4,True,12020.00
...,...,...,...
10507,13995,True,10.91
10508,13996,True,13231.82
10509,13997,True,1771.94
10510,13998,True,8670.77


In [27]:
c_df[~c_df.prism_consumer_id.isin(income_df.prism_consumer_id)]

,prism_consumer_id,evaluation_date,credit_score,DQ_TARGET
18,18,2021-01-01,806.0,0.0
20,20,2022-03-01,623.0,0.0
31,31,2021-08-01,659.0,0.0
34,34,2022-04-01,734.0,0.0
37,37,2021-10-01,738.0,0.0
...,...,...,...,...
13802,13802,2022-03-17,695.0,0.0
13807,13807,2021-12-14,689.0,0.0
13904,13904,2022-02-19,636.0,0.0
13917,13917,2021-12-01,627.0,0.0


### Seeing income and spending ratio/difference:

- Method 1:

In [28]:
diff_df = t_df.groupby(['prism_consumer_id', 'is_income'])[['amount']].sum().reset_index()
diff_df = diff_df.pivot(index='prism_consumer_id', columns='is_income', values='amount').fillna(0)

In [29]:
diff_df['difference'] = diff_df.get(True, 0) - diff_df.get(False, 0)
diff_df = diff_df.reset_index().rename_axis(None, axis=1)
diff_df

,prism_consumer_id,False,True,difference
0,0,20474.67,8820.56,-11654.11
1,1,36083.53,11918.64,-24164.89
2,2,45099.29,0.00,-45099.29
3,3,32739.45,9747.81,-22991.64
4,4,20455.82,12020.00,-8435.82
...,...,...,...,...
11596,13995,2542.28,2.23,-2540.05
11597,13996,95940.48,11702.26,-84238.22
11598,13997,13126.44,1771.94,-11354.50
11599,13998,92667.87,8158.17,-84509.70


- Method 2:

In [30]:
income_diff = income_df[['prism_consumer_id', 'amount']].merge(diff_df[['prism_consumer_id', False]], on='prism_consumer_id', how='left')
income_diff['difference'] = income_diff.get('amount', 0) - income_diff.get(False, 0)
income_diff = income_diff.rename_axis(None, axis=1)
income_diff

,prism_consumer_id,amount,False,difference
0,0,8860.56,20474.67,-11614.11
1,1,11918.64,36083.53,-24164.89
2,2,60.34,45099.29,-45038.95
3,3,10147.81,32739.45,-22591.64
4,4,12020.00,20455.82,-8435.82
...,...,...,...,...
10507,13995,10.91,2542.28,-2531.37
10508,13996,13231.82,95940.48,-82708.66
10509,13997,1771.94,13126.44,-11354.50
10510,13998,8670.77,92667.87,-83997.10


### Why some consumers aren't in the income df but present in the transaction df?:

In [31]:
display(diff_df[~diff_df.prism_consumer_id.isin(income_diff.prism_consumer_id)])
diff_df[~diff_df.prism_consumer_id.isin(income_diff.prism_consumer_id)][True].sum()

,prism_consumer_id,False,True,difference
18,18,223.66,0.0,-223.66
20,20,2000.03,0.0,-2000.03
31,31,2000.00,0.0,-2000.00
34,34,500.00,0.0,-500.00
37,37,1135.97,0.0,-1135.97
...,...,...,...,...
11403,13802,71255.07,0.0,-71255.07
11408,13807,57083.87,0.0,-57083.87
11505,13904,8092.11,0.0,-8092.11
11518,13917,495.66,0.0,-495.66


0.0

- They have no income based on the categories their transactions belong to

## Taking account balance into consideration:

In [32]:
display(a_df)
(a_df.balance < 0).sum()

,prism_consumer_id,prism_account_id,account_type,balance_date,balance
0,3023,0,SAVINGS,2021-08-31,90.57
1,3023,1,CHECKING,2021-08-31,225.95
5,3920,5,SAVINGS,2021-10-31,0.26
6,3920,6,CHECKING,2021-10-31,4.42
7,3920,7,CHECKING,2021-10-31,1.05
...,...,...,...,...,...
24461,11500,24461,CHECKING,2022-03-27,732.75
24462,11615,24462,SAVINGS,2022-03-30,5.00
24463,11615,24463,CHECKING,2022-03-30,1956.46
24464,12210,24464,CHECKING,2022-03-28,2701.51


476

- There are some instances where a bank balance is below zero (476 of them)

In [33]:
a_df.dtypes

prism_consumer_id             int32
prism_account_id              int32
account_type                 object
balance_date         datetime64[ns]
balance                     float64
dtype: object

In [34]:
a_df = a_df.sort_values(['prism_consumer_id', 'prism_account_id', 'account_type', 'balance_date'], ignore_index=True)
a_df

,prism_consumer_id,prism_account_id,account_type,balance_date,balance
0,0,862,SAVINGS,2021-08-31,25.70
1,0,863,CHECKING,2021-08-31,294.67
2,1,7754,SAVINGS,2021-06-30,3211.18
3,1,7755,CHECKING,2021-06-30,91.24
4,2,4666,SAVINGS,2021-04-30,2561.43
...,...,...,...,...,...
19482,13998,19885,CHECKING,2022-01-30,476.85
19483,13998,19936,LOAN,2022-01-30,252.93
19484,13998,19960,CREDIT CARD,2022-01-30,155.25
19485,13999,24213,SAVINGS,2022-01-26,39.01


In [35]:
a_df.groupby(['prism_consumer_id', 'prism_account_id', 'account_type']).agg(
    {'balance':
     [
         'sum', 
        #  'count'
    ]
    })

balance
                                                     sum
prism_consumer_id prism_account_id account_type         
0                 862              SAVINGS         25.70
                  863              CHECKING       294.67
1                 7754             SAVINGS       3211.18
                  7755             CHECKING        91.24
2                 4666             SAVINGS       2561.43
...                                                  ...
13998             19885            CHECKING       476.85
                  19936            LOAN           252.93
                  19960            CREDIT CARD    155.25
13999             24213            SAVINGS         39.01
                  24225            CHECKING       -80.01

[19487 rows x 1 columns]

## Metrics:
1. R and R^2

2. Point-Biserial Correlation:
    
        
    Positive: When the binary variable is 1, the continuous variable tends to have larger values.
    - Example: Higher test scores might correlate with being in the "Passed" (binary = 1) group.


    Negative: When the binary variable is 0, the continuous variable tends to have larger values.
    - Example: Lower income might correlate with not owning a house (binary = 0).

In [36]:
corr_df = pd.DataFrame(list(y_train.items()), columns=['prism_consumer_id', 'DQ_TARGET'])
corr_df['prism_consumer_id'] = corr_df['prism_consumer_id'].astype(diff_df['prism_consumer_id'].dtype)
corr_df = diff_df.merge(corr_df, on='prism_consumer_id', how='left')

display(
    # corr_df, 
    corr_df.corr(), 
    corr_df.corr()**2
)

,prism_consumer_id,False,True,difference,DQ_TARGET
prism_consumer_id,1.000000,0.136381,0.149127,-0.116947,0.050806
False,0.136381,1.000000,0.356851,-0.986524,-0.032637
True,0.149127,0.356851,1.000000,-0.199196,-0.048136
difference,-0.116947,-0.986524,-0.199196,1.000000,0.025806
DQ_TARGET,0.050806,-0.032637,-0.048136,0.025806,1.000000


,prism_consumer_id,False,True,difference,DQ_TARGET
prism_consumer_id,1.000000,0.018600,0.022239,0.013677,0.002581
False,0.018600,1.000000,0.127343,0.973229,0.001065
True,0.022239,0.127343,1.000000,0.039679,0.002317
difference,0.013677,0.973229,0.039679,1.000000,0.000666
DQ_TARGET,0.002581,0.001065,0.002317,0.000666,1.000000


In [37]:
r, p_value = pointbiserialr(corr_df.DQ_TARGET, corr_df.difference)
print("Point-Biserial Correlation:", r)
print("P-Value:", p_value)

Point-Biserial Correlation: 0.02580614250722851
P-Value: 0.005441084925950148


In [38]:
income_corr_df = pd.DataFrame(list(y_train.items()), columns=['prism_consumer_id', 'DQ_TARGET'])
income_corr_df['prism_consumer_id'] = income_corr_df['prism_consumer_id'].astype(income_diff['prism_consumer_id'].dtype)
income_corr_df = corr_df.merge(income_corr_df, on='prism_consumer_id', how='left')

display(
    # income_corr_df, 
    income_corr_df.corr(), 
    income_corr_df.corr()**2
)

,prism_consumer_id,False,True,difference,DQ_TARGET_x,DQ_TARGET_y
prism_consumer_id,1.000000,0.136381,0.149127,-0.116947,0.050806,0.050806
False,0.136381,1.000000,0.356851,-0.986524,-0.032637,-0.032637
True,0.149127,0.356851,1.000000,-0.199196,-0.048136,-0.048136
difference,-0.116947,-0.986524,-0.199196,1.000000,0.025806,0.025806
DQ_TARGET_x,0.050806,-0.032637,-0.048136,0.025806,1.000000,1.000000
DQ_TARGET_y,0.050806,-0.032637,-0.048136,0.025806,1.000000,1.000000


,prism_consumer_id,False,True,difference,DQ_TARGET_x,DQ_TARGET_y
prism_consumer_id,1.000000,0.018600,0.022239,0.013677,0.002581,0.002581
False,0.018600,1.000000,0.127343,0.973229,0.001065,0.001065
True,0.022239,0.127343,1.000000,0.039679,0.002317,0.002317
difference,0.013677,0.973229,0.039679,1.000000,0.000666,0.000666
DQ_TARGET_x,0.002581,0.001065,0.002317,0.000666,1.000000,1.000000
DQ_TARGET_y,0.002581,0.001065,0.002317,0.000666,1.000000,1.000000


In [39]:
r, p_value = pointbiserialr(corr_df.DQ_TARGET, corr_df.difference)
print("Point-Biserial Correlation:", r)
print("P-Value:", p_value)

Point-Biserial Correlation: 0.02580614250722851
P-Value: 0.005441084925950148


## Interpretation:
- No strong correlation between my feature (income - spending) and the binary variable, DQ_TARGET

This may mean:
- Relationship may not be linear
- Feature wasn't complex enough

# Predicting:

In [40]:
income_diff.prism_consumer_id.nunique()

10512

In [41]:
c_df[~c_df.prism_consumer_id.isin(income_diff.prism_consumer_id.unique())]['prism_consumer_id'].nunique()

1488

In [42]:
income_diff

,prism_consumer_id,amount,False,difference
0,0,8860.56,20474.67,-11614.11
1,1,11918.64,36083.53,-24164.89
2,2,60.34,45099.29,-45038.95
3,3,10147.81,32739.45,-22591.64
4,4,12020.00,20455.82,-8435.82
...,...,...,...,...
10507,13995,10.91,2542.28,-2531.37
10508,13996,13231.82,95940.48,-82708.66
10509,13997,1771.94,13126.44,-11354.50
10510,13998,8670.77,92667.87,-83997.10


In [43]:
income_diff = c_df[['prism_consumer_id', 'DQ_TARGET']].merge(income_diff, on='prism_consumer_id', how='left')
income_diff

,prism_consumer_id,DQ_TARGET,amount,False,difference
0,0,0.0,8860.56,20474.67,-11614.11
1,1,0.0,11918.64,36083.53,-24164.89
2,2,0.0,60.34,45099.29,-45038.95
3,3,0.0,10147.81,32739.45,-22591.64
4,4,0.0,12020.00,20455.82,-8435.82
...,...,...,...,...,...
11995,13995,0.0,10.91,2542.28,-2531.37
11996,13996,0.0,13231.82,95940.48,-82708.66
11997,13997,0.0,1771.94,13126.44,-11354.50
11998,13998,0.0,8670.77,92667.87,-83997.10


- The consumers that don't have transaction data:
    - Some are actually deemed as non_DQ

In [44]:
income_diff[income_diff.difference.isna()].DQ_TARGET.sum()

140.0

- Data splits:

In [45]:
cids = c_df.prism_consumer_id.unique()
print(len(cids))
train_cids, test_cids = train_test_split(cids, test_size=0.3, random_state=420) # 70 / 30 split

# Features and labels for train and test sets:
income_diff.loc[income_diff.difference.isna(), 'difference'] = 0.0
X_train = income_diff[income_diff.prism_consumer_id.isin(train_cids)]
y_train = X_train.DQ_TARGET
X_test  = income_diff[income_diff.prism_consumer_id.isin(test_cids)]
y_test  = X_test.DQ_TARGET

X_train.drop(columns=['prism_consumer_id', 'DQ_TARGET'], inplace=True)
X_test.drop(columns=['prism_consumer_id', 'DQ_TARGET'], inplace=True)

len(X_train), len(X_test)

12000


C:\Users\bdion\AppData\Local\Temp\ipykernel_1636\3015232756.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train.drop(columns=['prism_consumer_id', 'DQ_TARGET'], inplace=True)
C:\Users\bdion\AppData\Local\Temp\ipykernel_1636\3015232756.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test.drop(columns=['prism_consumer_id', 'DQ_TARGET'], inplace=True)


(8400, 3600)

In [46]:
clf = LogisticRegression(random_state=0, penalty=None).fit(np.array(X_train.difference).reshape(-1, 1), y_train)

y_pred = clf.predict(np.array(X_test.difference).reshape(-1,1))
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.92      0.97      0.94      3303
         1.0       0.03      0.01      0.02       297

    accuracy                           0.89      3600
   macro avg       0.47      0.49      0.48      3600
weighted avg       0.84      0.89      0.87      3600



In [47]:
y_pred.sum()

89.0

In [48]:
y_test.sum()

297.0